In [1]:
from haystack.components.converters import TextFileToDocument

In [2]:
from haystack.document_stores.in_memory import InMemoryDocumentStore 
from pathlib import Path
document_store = InMemoryDocumentStore()
converter = TextFileToDocument()
docs = converter.run(sources=[Path("case_query/example1.txt"), Path("case_query/example2.txt"), Path("case_query/example3.txt"), Path("case_query/example4.txt"), Path("case_query/example5.txt"), Path("case_query/example6.txt")])["documents"]

In [3]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

c:\Users\Yassamin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

Batches: 100%|██████████| 1/1 [00:00<00:00, 31.28it/s]


6

# Pipeline

In [5]:
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")

In [6]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store, top_k=1)

In [7]:
#XML Loader Component
from haystack import component
from pathlib import Path
from typing import Optional

@component
class XMLLoader:
    """
    Always loads the matching XML from case_document file based on the retrieved case_query document.
    If override_bpmn is also provided, it is appended as the new target BPMN to reason over.
    """
    
    @component.output_types(bpmn=str)
    def run(self, documents: list, override_bpmn: Optional[str] = None):
        
        if not documents:
            return {"bpmn": ""}
        
        qa_file = documents[0].meta.get('file_path', '')
        if not qa_file:
            return {"bpmn": ""}
        
        xml_path = Path(f"case_document/{Path(qa_file).name}")
        
        try:
            with open(xml_path, encoding='utf-8') as f:
                case_bpmn = f.read()
        except FileNotFoundError:
            print(f"XML file not found: {xml_path}")
            case_bpmn = ""
        
        # If a new BPMN is provided, it gets appended as the target process to reason over
        if override_bpmn and override_bpmn.strip():
            return {"bpmn": case_bpmn + "\n\nNew BPMN to answer the question from:\n" + override_bpmn}
        
        return {"bpmn": case_bpmn}

In [8]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage


template = [
    ChatMessage.from_user(
        """
         You are a helpful airport assistant answering questions from passengers. You speak in plain, friendly language, the same way a knowledgeable 
        airport employee would explain something to a traveller at an information desk at the airport.
        
        You will be given two inputs:
        - A BPMN process model "Example" serialized in XML: Use it to ground your answer, and try to find a pattern in the BPMN that matches the "Example Answer".
        - An "Example Question" and "Example Answer": these show you the tone, style, and level of detail of your answer, and the pattern in the "Example Answer".
        
        Hard Rules you must always follow:
        - Never mention BPMN, XML, task IDs, element IDs, or technical notation of any kind.
        - Do not use any quotation marks.
        - Your answer can't be longer than 3 sentences.
        - Never use the words "data object", "sequence flow", "gateway", "association", or any other BPMN term.
        - Answer only the question asked. Do not describe the overall process.
        - Write in plain prose. No bullet points, no headers, no lists.
        - If the user's question closely matches the Example Question, use the Example Answer as your response.
        
        BPMN: {{bpmn}}
        Question: {{question}}
        Answer:
        """
    )
]

prompt_builder = ChatPromptBuilder(template=template)


ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [9]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
chat_generator = OllamaChatGenerator(model="llama3.1", timeout=2000, url="http://localhost:11434", generation_kwargs={"num_ctx": 8192,"temperature": 0})

In [10]:
from haystack import Pipeline

basic_rag_pipeline = Pipeline()

basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("xml_loader", XMLLoader())
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", chat_generator)
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever.documents", "xml_loader.documents")
basic_rag_pipeline.connect("xml_loader.bpmn", "prompt_builder.bpmn")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - xml_loader: XMLLoader
  - prompt_builder: ChatPromptBuilder
  - llm: OllamaChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> xml_loader.documents (list[Document])
  - xml_loader.bpmn -> prompt_builder.bpmn (str)
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [ ]:
# Only paste the BPMN XML code (without the dimension part) here for the question category pattern adaptation. So in the cases, where the bpmn is not already in the case base.
#The following bpmns are not in the case base: 03_Departure_Boarding_Gate_International.bpmn, 04_Arrival_Lost_Bagggage.bpmn, 02_Departure_Baggage_Handover.bpmn
new_bpmn = """" """

In [13]:
question = "Why do I hand in my lost Baggage Form?"

response = basic_rag_pipeline.run(
    data={"text_embedder": {"text": question}, "prompt_builder": {"question": question}, "xml_loader": {"override_bpmn": new_bpmn}},
    include_outputs_from=["retriever"]
)


retrieved_docs = response["retriever"]["documents"]
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Relevance Score: {doc.score:.4f}") #cosine similarity score
    print(f"Source: {doc.meta.get('file_path', 'Unknown')}")

print(response["llm"]["replies"][0].text)

Batches: 100%|██████████| 1/1 [00:00<00:00, 118.49it/s]


Relevance Score: 0.5702
Source: example6.txt
You hand in your lost Baggage Form, so that it can be logged into the system by the Agent.
